In [73]:
import pandas as pd
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from pathlib import Path
import re
import string


from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

stop_words = set(stopwords.words('english'))
lem = WordNetLemmatizer()

# 1. DOWNLOAD REQUIRED NLTK RESOURCE

nltk.download("vader_lexicon", quiet=True)
nltk.download('punkt_tab')

# 2. CONFIGURATION

# Use r before the path to create a RAW STRING
INPUT_FILE = Path(
    r"C:\Users\AadharJain\Desktop\IBM_Training-main\DAY_5\unclean_customer_feedback_200.csv"
)

# Create output folder inside Machine Learning folder
OUTPUT_DIR = Path(
    r"C:\Users\AadharJain\Desktop\IBM_Training-main\DAY_5\sentiment_results"
)

# Create folder if it doesn't exist
OUTPUT_DIR.mkdir(exist_ok=True)


# Output file paths
POSITIVE_FILE = OUTPUT_DIR / "positive_feedback.csv"
NEGATIVE_FILE = OUTPUT_DIR / "negative_feedback.csv"
NEUTRAL_FILE = OUTPUT_DIR / "neutral_feedback.csv"
ALL_RESULTS_FILE = OUTPUT_DIR / "all_sentiment_results.csv"

# 3. LOAD DATA

df = pd.read_csv(INPUT_FILE)

print("Dataset loaded successfully!")
print(f"Total records: {len(df)}")

print("\nColumns available:")
print(df.columns.tolist())

Dataset loaded successfully!
Total records: 180

Columns available:
['Review_ID', 'Customer_Name', 'Email', 'Feedback']


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\AadharJain\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\AadharJain\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\AadharJain\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\AadharJain\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\AadharJain\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [74]:
df.head(2)

,Review_ID,Customer_Name,Email,Feedback
0,1001,John Smith,john.smith@gmail.com,I LOVE this product!!! It works PERFECTLY 😊. V...
1,1002,Emily Davis,emily@gmail.com,<div>Amazing quality!!!</div> I am VERY happy ...


In [75]:
# 4. VALIDATE REQUIRED COLUMN

TEXT_COLUMN = "Feedback"

if TEXT_COLUMN not in df.columns:
    raise ValueError(
        f"Column '{TEXT_COLUMN}' not found in the input file."
    )

In [ ]:
def preprocess(text, verbose=False):
    steps = []
    
    # Step 1: Lowercase
    text = text.lower()
    if verbose: steps.append(("1. Lowercase", text))
    
    # Step 2: Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    if verbose: steps.append(("2. Remove URLs", text))
    
    # Step 3: Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    if verbose: steps.append(("3. Remove HTML", text))
    
    # Step 4: Remove special characters

    # text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    # text = text.translate(str.maketrans('', '', string.punctuation))    
    # if verbose: steps.append(("4. Remove special chars", text))
    
    # Step 5: Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    if verbose: steps.append(("5. Normalize spaces", text))
    
    # Step 6: Tokenize
    tokens = nltk.word_tokenize(text)
    if verbose: steps.append(("6. Tokenize", str(tokens)))
    
    # Step 7: Remove stop words
    tokens = [t for t in tokens if t not in stop_words]
    if verbose: steps.append(("7. Remove stop words", str(tokens)))
    
    # Step 8: Lemmatize
    tokens = [lem.lemmatize(t, pos='v') for t in tokens]
    if verbose: steps.append(("8. Lemmatize", str(tokens)))
    
    if verbose:
        for step, result in steps:
            print(f"{step:<25}: {result}")
        print()
    
    return ' '.join(tokens)


In [77]:
# 5. HANDLE MISSING VALUES

df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str)
df[TEXT_COLUMN] = df[TEXT_COLUMN].apply(preprocess)

In [78]:
df.head(2)

,Review_ID,Customer_Name,Email,Feedback
0,1001,John Smith,john.smith@gmail.com,love product ! ! ! work perfectly 😊 . visit in...
1,1002,Emily Davis,emily@gmail.com,amaze quality ! ! ! happy purchase 😍😍


In [79]:
# 6. INITIALIZE SENTIMENT ANALYZER

sid = SentimentIntensityAnalyzer()

# 7. SENTIMENT ANALYSIS FUNCTION

def analyze_sentiment(text):

    # Get sentiment scores
    scores = sid.polarity_scores(text)

    # Extract compound score
    compound_score = scores["compound"]

    # Classify sentiment
    if compound_score >= 0.05:
        sentiment = "Positive"

    elif compound_score <= -0.05:
        sentiment = "Negative"

    else:
        sentiment = "Neutral"

    return pd.Series({
        "Positive_Score": scores["pos"],
        "Negative_Score": scores["neg"],
        "Neutral_Score": scores["neu"],
        "Compound_Score": compound_score,
        "Sentiment": sentiment
    })

# 8. APPLY SENTIMENT ANALYSIS

sentiment_results = df[TEXT_COLUMN].apply(
    analyze_sentiment
)


# Add sentiment results to original DataFrame
df = pd.concat(
    [df, sentiment_results],
    axis=1
)

# 9. CREATE SEPARATE DATASETS

positive_df = df[
    df["Sentiment"] == "Positive"
].copy()


negative_df = df[
    df["Sentiment"] == "Negative"
].copy()


neutral_df = df[
    df["Sentiment"] == "Neutral"
].copy()


# 10. SAVE RESULTS

positive_df.to_csv(
    POSITIVE_FILE,
    index=False
)

negative_df.to_csv(
    NEGATIVE_FILE,
    index=False
)

neutral_df.to_csv(
    NEUTRAL_FILE,
    index=False
)

df.to_csv(
    ALL_RESULTS_FILE,
    index=False
)

# 11. DISPLAY SUMMARY

print("\n" + "=" * 50)
print("SENTIMENT ANALYSIS SUMMARY")
print("=" * 50)

print(f"Total Reviews  : {len(df)}")
print(f"Positive       : {len(positive_df)}")
print(f"Negative       : {len(negative_df)}")
print(f"Neutral        : {len(neutral_df)}")


print("\nSentiment Distribution:")
print(
    df["Sentiment"].value_counts()
)


# 12. OUTPUT FILE LOCATIONS

print("\n" + "=" * 50)
print("FILES CREATED")
print("=" * 50)

print(f"Positive Reviews : {POSITIVE_FILE}")
print(f"Negative Reviews : {NEGATIVE_FILE}")
print(f"Neutral Reviews  : {NEUTRAL_FILE}")
print(f"All Results      : {ALL_RESULTS_FILE}")


SENTIMENT ANALYSIS SUMMARY
Total Reviews  : 180
Positive       : 89
Negative       : 44
Neutral        : 47

Sentiment Distribution:
Sentiment
Positive    89
Neutral     47
Negative    44
Name: count, dtype: int64

FILES CREATED
Positive Reviews : C:\Users\AadharJain\Desktop\IBM_Training-main\DAY_5\sentiment_results\positive_feedback.csv
Negative Reviews : C:\Users\AadharJain\Desktop\IBM_Training-main\DAY_5\sentiment_results\negative_feedback.csv
Neutral Reviews  : C:\Users\AadharJain\Desktop\IBM_Training-main\DAY_5\sentiment_results\neutral_feedback.csv
All Results      : C:\Users\AadharJain\Desktop\IBM_Training-main\DAY_5\sentiment_results\all_sentiment_results.csv
